In [ ]:
# import needed libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

### Exploration of Data Analysis:

In [ ]:
# Load dataset
titanic_data = pd.read_csv("Titanic-Dataset.csv")

In [ ]:
# info of dataset
print(titanic_data.info())

In [ ]:
# summary statistics
print(titanic_data.describe())

In [ ]:
# First few rows of the dataset
print(titanic_data.head())

### Data Cleansing:

In [ ]:
# Check missing values
titanic_data.isnull().sum()

In [ ]:
# In Age column fill missing value with the mean
titanic_data['Age'].fillna(titanic_data['Age'].median(), inplace=True)

# Convert 'Age' column to integer type
titanic_data['Age'] = titanic_data['Age'].astype(int)

In [ ]:
# Drop rows with missing Embarked values
titanic_data.dropna(subset=['Embarked'], inplace=True)

In [ ]:
# Drop irrelevant columns
titanic_data.drop(['Cabin', 'Ticket'], axis=1, inplace=True)

In [ ]:
# Check missing values again
titanic_data.isnull().sum()

### Data Visualization:

In [ ]:
# distribution of age
sns.histplot(titanic_data['Age'], bins=20, kde=True)
plt.title('Distribution of Age')
plt.show()

In [ ]:
# survival based on Pclass
sns.countplot(x='Pclass', hue='Survived', data=titanic_data)
plt.title('Survival based on Pclass')
plt.show()

In [ ]:
# survival based on Sex
sns.countplot(x='Sex', hue='Survived', data=titanic_data)
plt.title('Survival based on Sex')
plt.show()

In [ ]:
# Visualize correlation matrix
numeric_columns = titanic_data.select_dtypes(include=[np.number])
correlation_matrix = numeric_columns.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

### Modeling:

In [ ]:
# Select features and target variable
X = titanic_data[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = titanic_data['Survived']

# Convert categorical variables to numerical 
X = pd.get_dummies(X)

# Split the dataset into training and testing 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
model = RandomForestClassifier()

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

In [ ]:
# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

### Model Evaluation (Clustering - KMeans):

In [ ]:
# Select features for clustering
X_cluster = titanic_data[['Pclass', 'Age', 'Fare']]

# Standardize the features
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Determine the optimal number of clusters using the Elbow method
inertia = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertia.append(kmeans.inertia_)


In [ ]:
# Plot the Elbow method
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.show()


In [ ]:
# Based on the elbow method, choose the optimal number of clusters
optimal_k = 3

# Perform KMeans clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
titanic_data['Cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Evaluate clustering using silhouette score
silhouette_avg = silhouette_score(X_cluster_scaled, titanic_data['Cluster'])
print("Silhouette Score:", silhouette_avg)